hmm 3 class

In [3]:
import re
import numpy as np
import pandas as pd
from hmmlearn.hmm import CategoricalHMM
from sklearn.metrics import classification_report, f1_score

train = pd.read_csv("../data/train_data.csv", low_memory=False)
test = pd.read_csv("../data/test_data.csv", low_memory=False)

num_samples = 200000
num_val = 20000
docs_per_class = 15000
tune_metric = "macro"

train = train.dropna(subset=["overall"]).sample(num_samples, random_state=42)


def make_label(score):
    if score >= 4:
        return 2
    if score == 3:
        return 1
    return 0


def clean_words(text):
    text = str(text).lower().replace("n't", " not")
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()[:100]


def make_text(df):
    return df["summary"].fillna("") + " " + df["reviewText"].fillna("")


train["label"] = train["overall"].apply(make_label)
train["words"] = make_text(train).apply(clean_words)
test["words"] = make_text(test).apply(clean_words)

count = {}
for words in train["words"]:
    for w in words:
        count[w] = count.get(w, 0) + 1

vocab = {"unk": 0}
for w, c in count.items():
    if c >= 5:
        vocab[w] = len(vocab)


def to_ids(words):
    ids = [vocab.get(w, 0) for w in words]
    if len(ids) == 0:
        ids = [0]
    return np.array(ids).reshape(-1, 1)


train_ids = [to_ids(w) for w in train["words"]]
train_labels = list(train["label"])
test_ids = [to_ids(w) for w in test["words"]]

val_ids = train_ids[:num_val]
val_labels = train_labels[:num_val]
models = {}
for c in range(3):
    docs = [ids for ids, y in zip(train_ids[num_val:], train_labels[num_val:]) if y == c][:docs_per_class]
    print("class", c, "docs:", len(docs))
    model = CategoricalHMM(n_components=3, n_iter=10, n_features=len(vocab),
                           random_state=42, emissionprob_prior=1.1)
    model.fit(np.concatenate(docs), [len(d) for d in docs])
    models[c] = model


def get_scores(ids_list):
    all_scores = []
    for ids in ids_list:
        all_scores.append([models[c].score(ids) / len(ids) for c in range(3)])
    return all_scores


def predict_from_scores(row, bias):
    values = [row[c] + bias[c] for c in range(3)]
    return values.index(max(values))


val_scores = get_scores(val_ids)
half = len(val_scores) // 2
tune_scores = val_scores[:half]
tune_labels = val_labels[:half]
check_scores = val_scores[half:]
check_labels = val_labels[half:]

bias = [0.0, 0.0, 0.0]
pred = [predict_from_scores(r, bias) for r in tune_scores]
best_f1 = f1_score(tune_labels, pred, average=tune_metric)
steps = [-0.2, -0.1, -0.05, -0.02, -0.01, 0.0, 0.01, 0.02, 0.05, 0.1, 0.2]
for round_number in range(2):
    for c in [1, 2]:
        for step in steps:
            trial = bias[:]
            trial[c] = step
            pred = [predict_from_scores(r, trial) for r in tune_scores]
            f = f1_score(tune_labels, pred, average=tune_metric)
            if f > best_f1:
                best_f1 = f
                bias = trial
print("best bias:", bias)

before = [predict_from_scores(r, [0.0, 0.0, 0.0]) for r in check_scores]
after = [predict_from_scores(r, bias) for r in check_scores]
print("without bias micro f1:", f1_score(check_labels, before, average="micro"))
print("with bias micro f1:", f1_score(check_labels, after, average="micro"))
print(classification_report(check_labels, after, target_names=["dissatisfied", "neutral", "satisfied"]))

test_scores = get_scores(test_ids)
test_pred = [predict_from_scores(r, bias) for r in test_scores]
pd.DataFrame({"predicted": test_pred}).to_csv("q2_submission_hmm.csv", index=False)
print(pd.Series(test_pred).value_counts())

class 0 docs: 15000
class 1 docs: 15000
class 2 docs: 15000
best bias: [0.0, -0.01, 0.05]
without bias micro f1: 0.6632
with bias micro f1: 0.7837
              precision    recall  f1-score   support

dissatisfied       0.63      0.61      0.62      1635
     neutral       0.30      0.38      0.33       943
   satisfied       0.90      0.87      0.89      7422

    accuracy                           0.78     10000
   macro avg       0.61      0.62      0.61     10000
weighted avg       0.80      0.78      0.79     10000

2    9927
0    6007
1    4066
Name: count, dtype: int64


hmm 2 class

In [4]:
train = train[train["overall"] != 3]


def make_label(score):
    if score >= 4:
        return 1
    return 0


def clean_words(text):
    text = str(text).lower().replace("n't", " not")
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()[:100]


def make_text(df):
    return df["summary"].fillna("") + " " + df["reviewText"].fillna("")


train["label"] = train["overall"].apply(make_label)
train["words"] = make_text(train).apply(clean_words)
test["words"] = make_text(test).apply(clean_words)

count = {}
for words in train["words"]:
    for w in words:
        count[w] = count.get(w, 0) + 1

vocab = {"unk": 0}
for w, c in count.items():
    if c >= 5:
        vocab[w] = len(vocab)


def to_ids(words):
    ids = [vocab.get(w, 0) for w in words]
    if len(ids) == 0:
        ids = [0]
    return np.array(ids).reshape(-1, 1)


train_ids = [to_ids(w) for w in train["words"]]
train_labels = list(train["label"])
test_ids = [to_ids(w) for w in test["words"]]

val_ids = train_ids[:num_val]
val_labels = train_labels[:num_val]
models = {}
for c in range(2):
    docs = [ids for ids, y in zip(train_ids[num_val:], train_labels[num_val:]) if y == c][:docs_per_class]
    print("class", c, "docs:", len(docs))
    model = CategoricalHMM(n_components=3, n_iter=10, n_features=len(vocab),
                           random_state=42, emissionprob_prior=1.1)
    model.fit(np.concatenate(docs), [len(d) for d in docs])
    models[c] = model


def get_scores(ids_list):
    all_scores = []
    for ids in ids_list:
        all_scores.append([models[0].score(ids) / len(ids), models[1].score(ids) / len(ids)])
    return all_scores


def predict(row, bias):
    if row[1] + bias > row[0]:
        return 1
    return 0


val_scores = get_scores(val_ids)
half = len(val_scores) // 2
tune_scores = val_scores[:half]
tune_labels = val_labels[:half]
check_scores = val_scores[half:]
check_labels = val_labels[half:]

bias = 0.0
pred = [predict(r, bias) for r in tune_scores]
best_f1 = f1_score(tune_labels, pred, average=tune_metric)
steps = [-0.2, -0.1, -0.05, -0.02, -0.01, 0.0, 0.01, 0.02, 0.05, 0.1, 0.2]
for step in steps:
    pred = [predict(r, step) for r in tune_scores]
    f = f1_score(tune_labels, pred, average=tune_metric)
    if f > best_f1:
        best_f1 = f
        bias = step
print("best bias:", bias)

before = [predict(r, 0.0) for r in check_scores]
after = [predict(r, bias) for r in check_scores]
print("without bias micro f1:", f1_score(check_labels, before, average="micro"))
print("with bias micro f1:", f1_score(check_labels, after, average="micro"))
print(classification_report(check_labels, after, target_names=["dissatisfied", "satisfied"]))

test_scores = get_scores(test_ids)
test_pred = [predict(r, bias) for r in test_scores]
pd.DataFrame({"predicted": test_pred}).to_csv("q2_submission_hmm_2class.csv", index=False)
print(pd.Series(test_pred).value_counts())

class 0 docs: 15000
class 1 docs: 15000
best bias: 0.05
without bias micro f1: 0.841
with bias micro f1: 0.8925
              precision    recall  f1-score   support

dissatisfied       0.69      0.73      0.71      1814
   satisfied       0.94      0.93      0.93      8186

    accuracy                           0.89     10000
   macro avg       0.82      0.83      0.82     10000
weighted avg       0.90      0.89      0.89     10000

1    12020
0     7980
Name: count, dtype: int64
